# Column Level Security (CLS) -- `vstone_catalog.security`

**What is CLS?** Controls *which columns* (or column values) a user can see based on group membership.

**Why `vstone_catalog.security` schema?**
Same reason as RLS: governance views stored in a dedicated `security` schema cannot be
bypassed by querying Gold base tables directly. `SELECT` on base tables is revoked for
end users -- all access flows through `vstone_catalog.security`.

**Column sensitivity matrix:**

| Column | Visible to | Hidden / REDACTED for |
|---|---|---|
| `total_market_value_usd` | `admin_group`, `finance_group` | All others -> `**REDACTED**` |
| `avg_price_usd` | `admin_group`, `finance_group` | All others -> `**REDACTED**` |
| `price_rub` | `admin_group`, `finance_group` | All others -> `**REDACTED**` |
| `price_usd` | `admin_group`, `finance_group` | All others -> `**REDACTED**` |
| `price_per_hp_usd` | `admin_group`, `finance_group` | All others -> `**REDACTED**` |
| `color_r/g/b` | `admin_group` only | All others -> `NULL` |
| All other columns | Everyone | -- |


## Step 1 -- Inspect base Gold tables

In [0]:
-- ============================================================
-- STEP 1: Inspect base tables we will apply CLS to
-- ============================================================
SELECT * FROM vstone_catalog.gold.agg_top_10_brands_by_spend
ORDER BY total_market_value_usd DESC;

## Step 2 -- Confirm user identity and group membership

In [0]:
-- ============================================================
-- STEP 2: Confirm current user and group membership
-- finance_group  -> can see all price/revenue columns
-- admin_group    -> full access to all columns
-- others         -> price columns REDACTED
-- ============================================================
SELECT
  CURRENT_USER()                              AS current_user,
  IS_MEMBER('finance_group')                  AS is_finance,
  is_account_group_member('admin_group')      AS is_admin;

## Step 3 -- CLS view on `agg_top_10_brands_by_spend`
Revenue and price columns are `**REDACTED**` for users outside `finance_group` and `admin_group`.

In [0]:
-- ============================================================
-- STEP 3: CLS view on agg_top_10_brands_by_spend
-- Location: vstone_catalog.security.cls_brand_market_data
--
-- Sensitive columns:
--   total_market_value_usd  -> REDACTED for non-finance users
--   avg_price_usd           -> REDACTED for non-finance users
--
-- Non-sensitive (always visible):
--   brand, total_listings, gold_load_dt
--
-- CAST(...AS STRING) unifies the type so REDACTED and the
-- numeric value can coexist in a single STRING column.
-- For BI tools that need numeric type, finance_group users
-- should query the base view directly (granted separately).
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.cls_brand_market_data AS
SELECT
  brand,
  total_listings,

  -- Sensitive: total revenue -- visible only to finance_group or admin
  CAST(
    CASE
      WHEN is_account_group_member('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(total_market_value_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS total_market_value_usd,

  -- Sensitive: average price -- visible only to finance_group or admin
  CAST(
    CASE
      WHEN is_account_group_member('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(avg_price_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS avg_price_usd,

  gold_load_dt
FROM vstone_catalog.gold.agg_top_10_brands_by_spend;

## Step 4 -- CLS view on `fact_listings`
Three price columns redacted. RGB color columns excluded entirely for non-admin users.

In [0]:
-- ============================================================
-- STEP 4: CLS view on fact_listings (transaction grain)
-- Location: vstone_catalog.security.cls_fact_listings
--
-- Sensitive columns:
--   price_rub        -> REDACTED for non-finance users
--   price_usd        -> REDACTED for non-finance users
--   price_per_hp_usd -> REDACTED for non-finance users
--   color_r/g/b      -> excluded entirely for non-admin users
--                       (internal RGB tracking data)
--
-- Non-sensitive (always visible):
--   listing_id, brand, model, manufacture_year, listing_date,
--   price_category, mileage_km, fuel_type, transmission_type,
--   location_key, photo_count, car_age_at_listing, is_high_mileage
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.cls_fact_listings AS
SELECT
  -- Always visible columns (no sensitivity)
  listing_id,
  brand,
  model,
  manufacture_year,
  listing_date,
  price_category,       -- category label (not the raw price -- safe to expose)
  mileage_km,
  fuel_type,
  transmission_type,
  location_key,
  photo_count,
  car_age_at_listing,
  is_high_mileage,

  -- Sensitive: raw prices -- REDACTED for non-finance users
  CAST(
    CASE
      WHEN is_account_group_member('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(price_rub AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_rub,

  CAST(
    CASE
      WHEN is_account_group_member('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(price_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_usd,

  CAST(
    CASE
      WHEN is_account_group_member('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(price_per_hp_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_per_hp_usd,

  -- Sensitive: RGB color tracking -- excluded for non-admin users entirely
  CASE
    WHEN is_account_group_member('admin_group')
      THEN color_r ELSE NULL
  END AS color_r,
  CASE
    WHEN is_account_group_member('admin_group')
      THEN color_g ELSE NULL
  END AS color_g,
  CASE
    WHEN is_account_group_member('admin_group')
      THEN color_b ELSE NULL
  END AS color_b,

  -- Audit columns always visible
  silver_load_dt,
  gold_load_dt
FROM vstone_catalog.gold.fact_listings;

## Step 5 -- Verify masked output

In [0]:
-- ============================================================
-- STEP 5: Verify CLS views
-- finance_group users see numeric prices.
-- Other users see **REDACTED** in those columns.
-- ============================================================

-- Brand market data with column masking
SELECT * FROM vstone_catalog.security.cls_brand_market_data
ORDER BY total_listings DESC;

-- Fact listings with column masking (limit for display)
SELECT
  listing_id, brand, price_rub, price_usd,
  price_per_hp_usd, price_category, color_r
FROM vstone_catalog.security.cls_fact_listings
LIMIT 20;